# 03 | Univariate and bivariate EDA
This notebook builds separate, readable analysis blocks. Figures export to `reports/figures/`. Interpret differences with group sizes and missingness; do not expose identifiable employee data.

## 1. Load cleaned dataset

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
sns.set_theme(style="whitegrid")
df=pd.read_csv("../data/interim/cleaned_day2.csv")
FIG=Path("../reports/figures"); FIG.mkdir(parents=True,exist_ok=True)
print(df.shape)
display(df.head())

## 2. Numeric summary

In [ ]:
num=["salary","engagement_score","employee_satisfaction","absences","days_late_last_30"]
display(df[num].describe().T)

## 3. Histograms and KDEs

In [ ]:
for col in num:
    fig,ax=plt.subplots(figsize=(8,4))
    sns.histplot(df[col].dropna(),kde=True,ax=ax)
    ax.set_title(f"Distribution of {col}")
    fig.tight_layout(); fig.savefig(FIG/f"{col}_histogram.png",dpi=160); plt.show()

## 4. Boxplots

In [ ]:
for col in num:
    fig,ax=plt.subplots(figsize=(8,3))
    sns.boxplot(x=df[col],ax=ax); ax.set_title(f"{col}: spread and potential outliers")
    fig.tight_layout(); fig.savefig(FIG/f"{col}_boxplot.png",dpi=160); plt.show()

## 5. Categorical frequencies and charts

In [ ]:
cats=["department","job_title","recruitment_source","employment_status","performance_score","work_mode","sex","marital_status"]
for col in cats:
    counts=df[col].fillna("Missing").value_counts()
    if col=="job_title":
        counts=pd.concat([counts[counts>=5],pd.Series({"Other titles (<5)":int(counts[counts<5].sum())})])
    display(counts.rename_axis(col).to_frame("Count"))
    fig,ax=plt.subplots(figsize=(9,max(3,min(8,.32*len(counts)))))
    counts.sort_values().plot.barh(ax=ax); ax.set_title(f"{col} frequency")
    fig.tight_layout(); fig.savefig(FIG/f"{col}_bar.png",dpi=160); plt.show()

## 6. Numeric-numeric relationships

In [ ]:
fig,ax=plt.subplots(figsize=(8,6)); sns.heatmap(df[num].corr(),annot=True,cmap="vlag",center=0,ax=ax); ax.set_title("Numeric feature correlation")
fig.tight_layout(); fig.savefig(FIG/"correlation_heatmap.png",dpi=160); plt.show()

In [ ]:
for x,y in [("absences","employee_satisfaction"),("salary","engagement_score"),("salary","employee_satisfaction")]:
    fig,ax=plt.subplots(figsize=(7,4)); sns.scatterplot(data=df,x=x,y=y,alpha=.35,s=20,ax=ax)
    ax.set_title(f"{x} vs {y}"); fig.tight_layout(); fig.savefig(FIG/f"{x}_vs_{y}.png",dpi=160); plt.show()

## 7. Numeric-categorical comparisons

In [ ]:
for group in ["department","work_mode","employment_status"]:
    display(df.groupby(group,dropna=False)[num].agg(["count","median","mean"]).round(2))
fig,ax=plt.subplots(figsize=(10,5)); sns.boxplot(data=df,x="department",y="employee_satisfaction",ax=ax); ax.tick_params(axis="x",rotation=45); fig.tight_layout(); fig.savefig(FIG/"satisfaction_by_department.png",dpi=160); plt.show()

## 8. Segment findings prompt
Identify which segments have differences in center/spread, state sample sizes, and note missingness. Treat results as exploratory and avoid claims about individuals.

## Additional business-focused comparisons
Compare useful segments directly. Missing work mode is excluded only from the substantive association test, not removed from the source dataset. Termination reasons are limited to terminated employees. Tenure follows the shared AS_OF_DATE documented in Day 2.

In [ ]:
# Department vs employment status
ct=pd.crosstab(df.department,df.employment_status)
display(ct); display((ct.div(ct.sum(axis=1),axis=0)*100).round(1))
ct.plot(kind="bar",stacked=True,figsize=(10,5),title="Employment status by department");plt.tight_layout();plt.savefig(FIG/"status_by_department.png",dpi=160);plt.show()
# Recruitment channel by department
ct2=pd.crosstab(df.department,df.recruitment_source)
fig,ax=plt.subplots(figsize=(12,6));sns.heatmap(ct2,annot=True,fmt="d",cmap="Blues",ax=ax);ax.set_title("Recruitment source by department");fig.tight_layout();fig.savefig(FIG/"recruitment_by_department.png",dpi=160);plt.show()
# Termination reason: terminated records only
term=df[df.terminated.eq(1)&~df.termination_reason.eq("N/A - Still Employed")]
reason=term.termination_reason.value_counts();display(reason.to_frame("count"));reason.sort_values().plot.barh(figsize=(8,5),title="Termination reasons");plt.tight_layout();plt.savefig(FIG/"termination_reasons.png",dpi=160);plt.show()
# Satisfaction by work mode and attendance by employment status
sns.boxplot(data=df[df.work_mode.ne("Missing")],x="work_mode",y="employee_satisfaction");plt.title("Satisfaction by work mode");plt.tight_layout();plt.savefig(FIG/"satisfaction_by_work_mode.png",dpi=160);plt.show()
sns.boxplot(data=df,x="employment_status",y="absences");plt.xticks(rotation=25);plt.title("Absences by employment status");plt.tight_layout();plt.savefig(FIG/"absences_by_status.png",dpi=160);plt.show()
# Tenure and salary by job title
display(df.groupby("job_title").salary.agg(["count","median"]).query("count >= 5").sort_values("median",ascending=False).head(15))
sns.histplot(df.tenure_years.dropna(),kde=True);plt.title("Employee tenure (years)");plt.tight_layout();plt.savefig(FIG/"tenure_distribution.png",dpi=160);plt.show()
print("Spearman engagement-satisfaction",spearmanr(df.engagement_score,df.employee_satisfaction,nan_policy="omit"))